# Traffic Sign Recognition & Road Safety Vision Pipeline
### Multi-Model Benchmarking, Deep Learning Classifiers, Faster R-CNN, Segmentation & Conv Visualizations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shubhamjena28/traffic-sign-recognition-road-safety/blob/main/train/vision_pipeline_colab.ipynb)

This comprehensive Google Colab notebook implements the complete vision engineering roadmap:
1. **ResNet-18 Classifier** on GTSRB crops with data augmentation (10 epochs)
2. **GoogLeNet (Inception v1) Classifier** with Top-1 Accuracy, Confusion Matrix, and Per-Class Recall
3. **Faster R-CNN (ResNet-50 FPN)** Object Detector fine-tuning on road safety detection sets
4. **Semantic Segmentation (DeepLabV3)** on BDD100K road scenes reporting Mean IoU and Pixel Accuracy
5. **Convolution Visualizations**: 2D Conv Kernels, Feature Maps, 3D Conv on 5-frame video stack, and Instance Segmentation
6. **Unified Master Benchmark Table**: Evaluating mAP@0.5, mAP@0.5:0.95, IoU, Missed-Sign Rates, Parameters, and CPU Latency

## Step 1: Verify Hardware Acceleration (GPU)

In [ ]:
!nvidia-smi
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")

## Step 2: Clone Repository & Install Dependencies

In [ ]:
import os
if not os.path.exists("traffic-sign-recognition-road-safety"):
    !git clone https://github.com/shubhamjena28/traffic-sign-recognition-road-safety.git

%cd traffic-sign-recognition-road-safety

!pip install --quiet ultralytics opencv-python matplotlib pandas scikit-learn torchvision torch

## Step 3: Verify & Prepare Datasets

In [ ]:
# Step 3: Dataset Setup & Verification
# Checks for dataset presence and automatically generates starter images if cloning fresh
from pathlib import Path
import os

img_dir = Path('data/images/train')
train_imgs = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png')) if img_dir.exists() else []

if len(train_imgs) == 0:
    print('[INFO] Images not found in data/images/train (datasets are git-ignored).')
    print('[INFO] Generating starter dataset with Stop, Speed, and Danger signs...')
    !python data/create_sample_dataset.py

train_count = len(list(Path('data/images/train').glob('*.jpg')))
val_count = len(list(Path('data/images/val').glob('*.jpg')))
test_count = len(list(Path('data/images/test').glob('*.jpg')))
print(f'[✓] Dataset Ready: {train_count} train, {val_count} val, {test_count} test images in data/images')


### (Optional) Use Your Custom Dataset from Google Drive
If you uploaded your local `data/images` folder (from `C:\Users\HP\...\data\images`) to your Google Drive, run this cell to mount Drive and copy your dataset into Colab:

In [ ]:
# UNCOMMENT TO MOUNT GOOGLE DRIVE IF USING PERSONAL DATASET:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p data/images data/labels
# !cp -r '/content/drive/MyDrive/traffic-sign-data/images/'* data/images/
# !cp -r '/content/drive/MyDrive/traffic-sign-data/labels/'* data/labels/
# print(f'Copied custom dataset. Current images: {len(list(Path("data/images/train").glob("*.jpg")))}')

## Step 4: Train ResNet-18 Classifier on GTSRB Crops (with Augmentation)
Fine-tunes ResNet-18 with ColorJitter, RandomRotation, and RandomAffine.

In [ ]:
!python train/train_resnet18.py --epochs 10 --batch-size 32 --lr 0.001 --device auto

## Step 5: Train GoogLeNet (Inception v1) Classifier
Trains GoogLeNet on the same crops. Generates Confusion Matrix and Per-Class Recall.

In [ ]:
!python train/train_googlenet.py --epochs 10 --batch-size 32 --lr 0.001 --device auto

## Step 6: Fine-Tune Faster R-CNN (ResNet-50 FPN) Detector
Fine-tunes torchvision two-stage Faster R-CNN on the road sign detection set.

In [ ]:
!python train/train_faster_rcnn.py --epochs 5 --batch-size 4 --lr 0.0005 --device auto

## Step 7: Train Semantic Segmentation Model on BDD100K Subset
Trains DeepLabV3 on road scenes, reporting Mean IoU and Pixel Accuracy.

In [ ]:
!python train/train_segmentation.py --epochs 5 --batch-size 8 --lr 0.001 --device auto

## Step 8: Convolution Visualizations & Advanced Demos
Generates:
1. **2D Filter Kernels** (ResNet-18 conv1 7x7 spatial filters)
2. **Feature Maps** across ResNet-18 layers (conv1, layer1..4)
3. **3D Convolution Demo** on a 5-frame video approach stack (nn.Conv3d)
4. **Instance Segmentation Demo** with polygonal masks and boxes

In [ ]:
!python train/visualize_conv.py

# Display generated visualizations
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

fig_names = [
    "checkpoints/resnet_conv1_kernels.png",
    "checkpoints/resnet_feature_maps.png",
    "checkpoints/3d_conv_temporal_stack.png",
    "checkpoints/instance_segmentation_demo.png"
]

for f in fig_names:
    if Path(f).exists():
        plt.figure(figsize=(10, 6))
        plt.imshow(Image.open(f))
        plt.title(f)
        plt.axis("off")
        plt.show()

## Step 9: Comprehensive Unified Master Benchmark Table
Outputs mAP@0.5, mAP@0.5:0.95, IoU, missed-sign rates, params, and CPU ms/frame in ONE table.

In [ ]:
!python train/benchmark_all.py

with open("docs/benchmark_table.md") as f:
    from IPython.display import Markdown, display
    display(Markdown(f.read()))

## Step 10: Download Trained Models & Artifacts
Downloads checkpoints and visual plots directly to your computer.

In [ ]:
from google.colab import files
import glob

download_files = glob.glob("checkpoints/*.pt") + glob.glob("checkpoints/*.png")
for df in download_files:
    print(f"Downloading: {df}")
    files.download(df)